# Distance-Specific GraphSAGE Training

This notebook is a **training worker** that trains a GraphSAGE model on user-specified code distances.

**Usage:**
1. Set the `DISTANCES` parameter in the Configuration cell (e.g., `[3, 11]`)
2. Run all cells
3. Training artifacts are saved to `models/`, `results/`, and `plots/`

**Features:**
- Uses best hyperparameters from tuning experiments
- Trains for 50 epochs with comprehensive logging
- Saves model checkpoint, training metrics JSON, plots, and CSV summary
- Supports both local and Google Colab execution

## 1. Configuration (USER INPUT)

In [ ]:
# =============================================================================
# USER CONFIGURATION - MODIFY THIS CELL
# =============================================================================

# Specify which code distances to train on
# Examples:
#   DISTANCES = [3]           # Train only on d=3
#   DISTANCES = [3, 5, 7]     # Train on d=3, d=5, d=7
#   DISTANCES = [3, 11]       # Train on d=3 and d=11
#   DISTANCES = [5, 7, 9, 11] # Train on multiple larger distances

DISTANCES = [3, 11]  # <-- MODIFY THIS

# =============================================================================
# AUTO-GENERATED EXPERIMENT NAME
# =============================================================================

EXPERIMENT_NAME = "_".join([f"d{d}" for d in sorted(DISTANCES)])

print(f"Configuration:")
print(f"  Distances: {DISTANCES}")
print(f"  Experiment name: {EXPERIMENT_NAME}")

## 2. Imports and Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install stim pymatching numpy matplotlib torch tqdm networkx pandas
# !pip install torch_geometric

In [ ]:
import sys
import json
import random
import time
import platform
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/gSAGE/distances -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from torch_geometric.loader import DataLoader

# Import from models.py
from models import (
    SurfaceCodeSampler,
    SparseGraph,
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

# Set up paths for this experiment
DISTANCES_DIR = BASE_PATH / "gSAGE" / "distances"
MODELS_DIR = DISTANCES_DIR / "models"
RESULTS_DIR = DISTANCES_DIR / "results"
PLOTS_DIR = DISTANCES_DIR / "plots"

# Create output directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU_NAME}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    GPU_NAME = None

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  MODELS_DIR: {MODELS_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

## 3. Load Best Hyperparameters

In [ ]:
# =============================================================================
# LOAD BEST HYPERPARAMETERS FROM TUNING
# =============================================================================

BEST_CONFIG_PATH = BASE_PATH / "gSAGE" / "tuning" / "best_model_config.json"

with open(BEST_CONFIG_PATH, 'r') as f:
    best_config = json.load(f)

# Extract hyperparameters
BEST_HYPERPARAMS = {
    'num_layers': best_config['model_params']['num_layers'],
    'hidden_dim': best_config['model_params']['hidden_dim'],
    'learning_rate': best_config['training_params']['learning_rate'],
    'dropout': best_config['model_params']['dropout'],
    'aggr': best_config['model_params']['aggr'],
}

print(f"Loaded best hyperparameters from: {BEST_CONFIG_PATH}")
print(f"\nTuning results:")
print(f"  Config ID: {best_config['config_id']}")
print(f"  Validation accuracy: {best_config['performance']['val_accuracy']*100:.2f}%")
print(f"  Test accuracy: {best_config['performance']['test_accuracy']*100:.2f}%")
print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

## 4. Training Configuration

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Training parameters
EPOCHS = 50
BATCH_SIZE = 256

# Dataset parameters
TOTAL_SAMPLES = 1_000_000  # Total samples across all distances
SAMPLES_PER_DISTANCE = TOTAL_SAMPLES // len(DISTANCES)  # Equal split

# Error rate distribution
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Graph construction
K_NEIGHBORS = 6
IN_CHANNELS = 5  # Node features: [X?, Z?, d_North, d_West, d_time]

# Reproducibility
SEED = 42

# Dataset split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print(f"Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {BEST_HYPERPARAMS['learning_rate']}")
print(f"\nDataset Configuration:")
print(f"  Total samples: {TOTAL_SAMPLES:,}")
print(f"  Distances: {DISTANCES}")
print(f"  Samples per distance: {SAMPLES_PER_DISTANCE:,}")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")
print(f"\nSplit ratios:")
print(f"  Train: {TRAIN_RATIO*100:.0f}%")
print(f"  Validation: {VAL_RATIO*100:.0f}%")
print(f"  Test: {TEST_RATIO*100:.0f}%")

## 5. Dataset Generation/Loading

In [ ]:
def generate_or_load_dataset(d: int, n_samples: int, cache_name: str = None) -> list:
    """
    Generate or load a dataset for a specific distance.
    
    Args:
        d: Code distance
        n_samples: Number of samples to generate/load
        cache_name: Optional cache name to load from/save to
        
    Returns:
        List of PyG Data objects
    """
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    
    # Try to load from cache first
    if cache_name:
        try:
            cache.load(cache_name, verbose=True)
            if cache.size() >= n_samples:
                print(f"  Loaded {n_samples:,} samples from cache '{cache_name}'")
                return cache.get_graphs(n=n_samples, shuffle=True)
            else:
                print(f"  Cache has {cache.size():,} samples, need {n_samples:,}. Growing...")
                cache.ensure_size(n_samples, verbose=True)
                cache.save(cache_name)
                return cache.get_graphs(n=n_samples, shuffle=True)
        except FileNotFoundError:
            print(f"  Cache '{cache_name}' not found. Generating new dataset...")
    
    # Generate new dataset
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
        k_neighbors=K_NEIGHBORS,
        verbose=True
    )
    
    # Save to cache if name provided
    if cache_name:
        cache.save(cache_name)
    
    return cache.get_graphs(shuffle=True)

In [ ]:
def load_combined_dataset(distances: list, samples_per_distance: int) -> tuple:
    """
    Load and combine datasets for multiple distances.
    
    Args:
        distances: List of code distances
        samples_per_distance: Number of samples per distance
        
    Returns:
        Tuple of (combined_graphs, dataset_info)
    """
    print(f"\n{'='*60}")
    print(f"Loading datasets for distances: {distances}")
    print(f"Samples per distance: {samples_per_distance:,}")
    print(f"{'='*60}")
    
    combined_graphs = []
    dataset_info = {}
    
    for d in distances:
        cache_name = f"d{d}_baseline"
        print(f"\n[d={d}] Loading {samples_per_distance:,} samples...")
        
        start_time = time.time()
        graphs = generate_or_load_dataset(d, samples_per_distance, cache_name)
        load_time = time.time() - start_time
        
        combined_graphs.extend(graphs)
        dataset_info[f'd{d}'] = {
            'samples': len(graphs),
            'load_time_seconds': load_time
        }
        print(f"  Added {len(graphs):,} samples (took {load_time:.1f}s)")
    
    # Shuffle combined dataset
    print(f"\nShuffling combined dataset...")
    random.seed(SEED)
    random.shuffle(combined_graphs)
    
    print(f"\nCombined dataset: {len(combined_graphs):,} total samples")
    return combined_graphs, dataset_info

In [ ]:
def split_dataset(graphs: list, train_ratio: float, val_ratio: float, seed: int):
    """
    Split dataset into train/val/test sets.
    
    Args:
        graphs: List of PyG Data objects
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        seed: Random seed
        
    Returns:
        Tuple of (train_graphs, val_graphs, test_graphs)
    """
    random.seed(seed)
    indices = list(range(len(graphs)))
    random.shuffle(indices)
    
    n = len(graphs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    
    train_graphs = [graphs[i] for i in train_idx]
    val_graphs = [graphs[i] for i in val_idx]
    test_graphs = [graphs[i] for i in test_idx]
    
    print(f"\nDataset split:")
    print(f"  Train: {len(train_graphs):,} samples ({len(train_graphs)/n*100:.1f}%)")
    print(f"  Validation: {len(val_graphs):,} samples ({len(val_graphs)/n*100:.1f}%)")
    print(f"  Test: {len(test_graphs):,} samples ({len(test_graphs)/n*100:.1f}%)")
    
    return train_graphs, val_graphs, test_graphs

In [ ]:
# Load and prepare datasets
all_graphs, dataset_info = load_combined_dataset(DISTANCES, SAMPLES_PER_DISTANCE)
train_graphs, val_graphs, test_graphs = split_dataset(all_graphs, TRAIN_RATIO, VAL_RATIO, SEED)

## 6. Training with Heavy Logging

In [ ]:
def evaluate_model(model, graphs, device, batch_size=256):
    """
    Evaluate model accuracy on a set of graphs.
    
    Args:
        model: GraphSAGE model wrapper
        graphs: List of PyG Data objects
        device: Torch device
        batch_size: Batch size for evaluation
        
    Returns:
        Accuracy as a float
    """
    model.model.eval()
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model.model(batch)
            y = batch.y.float().view(-1, 1)
            correct += ((pred > 0.5).float() == y).sum().item()
            total += y.size(0)
    
    return correct / total if total > 0 else 0.0

In [ ]:
def train_with_logging(model, train_graphs, val_graphs, epochs, batch_size, lr, device, verbose=True):
    """
    Train model with comprehensive per-epoch logging.
    
    Args:
        model: GraphSAGE model wrapper
        train_graphs: Training data
        val_graphs: Validation data
        epochs: Number of epochs
        batch_size: Batch size
        lr: Learning rate
        device: Torch device
        verbose: Print progress
        
    Returns:
        Dict with per-epoch metrics
    """
    loader = DataLoader(
        train_graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    
    model.model.train()
    optimizer = torch.optim.Adam(model.model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()
    
    # Initialize logging containers
    epoch_metrics = {
        'epoch': [],
        'train_loss': [],
        'train_accuracy': [],
        'val_accuracy': [],
        'epoch_time_seconds': [],
        'learning_rate': [],
        'gpu_memory_mb': [],
    }
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training: {model.nickname}")
        print(f"{'='*70}")
        print(f"Epochs: {epochs} | Batch size: {batch_size} | LR: {lr}")
        print(f"Training samples: {len(train_graphs):,}")
        print(f"Validation samples: {len(val_graphs):,}")
        print(f"{'='*70}\n")
    
    total_start_time = time.time()
    
    for epoch in range(epochs):
        epoch_start_time = time.time()
        
        # Training phase
        model.model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", 
                    disable=not verbose, leave=False)
        
        for batch_data in pbar:
            batch_data = batch_data.to(device)
            pred = model.model(batch_data)
            y = batch_data.y.float().view(-1, 1)
            
            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Accumulate metrics
            epoch_loss += loss.item()
            epoch_correct += ((pred > 0.5).float() == y).sum().item()
            epoch_total += y.size(0)
            
            # Update progress bar
            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.1f}%'})
        
        pbar.close()
        
        # Calculate epoch metrics
        avg_train_loss = epoch_loss / len(loader)
        train_accuracy = epoch_correct / epoch_total
        
        # Validation phase
        val_accuracy = evaluate_model(model, val_graphs, device, batch_size)
        
        epoch_time = time.time() - epoch_start_time
        
        # GPU memory usage
        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0
        
        # Log metrics
        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_accuracy)
        epoch_metrics['val_accuracy'].append(val_accuracy)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['learning_rate'].append(lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        
        # Print epoch summary
        if verbose:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {avg_train_loss:.4f} | "
                  f"Train Acc: {train_accuracy*100:.2f}% | "
                  f"Val Acc: {val_accuracy*100:.2f}% | "
                  f"Time: {epoch_time:.1f}s | "
                  f"GPU: {gpu_memory_mb:.0f}MB")
    
    total_time = time.time() - total_start_time
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training complete!")
        print(f"Total time: {total_time/60:.1f} minutes")
        print(f"Final train loss: {epoch_metrics['train_loss'][-1]:.4f}")
        print(f"Final train accuracy: {epoch_metrics['train_accuracy'][-1]*100:.2f}%")
        print(f"Final val accuracy: {epoch_metrics['val_accuracy'][-1]*100:.2f}%")
        print(f"{'='*70}")
    
    epoch_metrics['total_training_time_seconds'] = total_time
    
    return epoch_metrics

In [ ]:
# Initialize model with best hyperparameters
print(f"\n{'='*70}")
print(f"Initializing GraphSAGE model")
print(f"{'='*70}")

model = GraphSAGE(
    nickname=f"distances_{EXPERIMENT_NAME}",
    in_channels=IN_CHANNELS,
    hidden_dim=BEST_HYPERPARAMS['hidden_dim'],
    num_layers=BEST_HYPERPARAMS['num_layers'],
    dropout=BEST_HYPERPARAMS['dropout'],
    aggr=BEST_HYPERPARAMS['aggr'],
    device=device,
    base_path=BASE_PATH,
    seed=SEED
)

# Count parameters
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

In [ ]:
# Train the model
epoch_metrics = train_with_logging(
    model=model,
    train_graphs=train_graphs,
    val_graphs=val_graphs,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=BEST_HYPERPARAMS['learning_rate'],
    device=device,
    verbose=True
)

In [ ]:
# Evaluate on test set
print(f"\n{'='*70}")
print(f"Final Evaluation")
print(f"{'='*70}")

# Subsample train for faster evaluation
train_eval_subset = train_graphs[:10000] if len(train_graphs) > 10000 else train_graphs

final_train_acc = evaluate_model(model, train_eval_subset, device, BATCH_SIZE)
final_val_acc = evaluate_model(model, val_graphs, device, BATCH_SIZE)
final_test_acc = evaluate_model(model, test_graphs, device, BATCH_SIZE)

print(f"Train accuracy (subset): {final_train_acc*100:.2f}%")
print(f"Validation accuracy: {final_val_acc*100:.2f}%")
print(f"Test accuracy: {final_test_acc*100:.2f}%")

## 7. Save All Artifacts

In [ ]:
# =============================================================================
# SAVE MODEL CHECKPOINT
# =============================================================================

model_path = MODELS_DIR / f"{EXPERIMENT_NAME}.pt"

model_save_dict = {
    'state_dict': model.model.state_dict(),
    'config': model._config,
    'experiment_name': EXPERIMENT_NAME,
    'distances': DISTANCES,
    'hyperparams': BEST_HYPERPARAMS,
    'training_config': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'total_samples': TOTAL_SAMPLES,
        'samples_per_distance': SAMPLES_PER_DISTANCE,
        'p_values': P_VALUES,
        'p_weights': P_WEIGHTS,
        'k_neighbors': K_NEIGHBORS,
        'seed': SEED,
    },
    'final_metrics': {
        'train_accuracy': final_train_acc,
        'val_accuracy': final_val_acc,
        'test_accuracy': final_test_acc,
        'final_loss': epoch_metrics['train_loss'][-1],
    },
    'timestamp': datetime.now().isoformat(),
}

torch.save(model_save_dict, model_path)
print(f"Model saved to: {model_path}")

In [ ]:
# =============================================================================
# SAVE TRAINING RESULTS JSON
# =============================================================================

results_path = RESULTS_DIR / f"{EXPERIMENT_NAME}_training.json"

training_results = {
    'experiment_name': EXPERIMENT_NAME,
    'distances': DISTANCES,
    'hyperparams': BEST_HYPERPARAMS,
    'training_config': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'total_samples': TOTAL_SAMPLES,
        'samples_per_distance': SAMPLES_PER_DISTANCE,
        'p_values': P_VALUES,
        'p_weights': P_WEIGHTS,
        'k_neighbors': K_NEIGHBORS,
        'seed': SEED,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
    },
    'dataset_info': dataset_info,
    'epoch_metrics': {
        'epochs': epoch_metrics['epoch'],
        'train_loss': epoch_metrics['train_loss'],
        'train_accuracy': epoch_metrics['train_accuracy'],
        'val_accuracy': epoch_metrics['val_accuracy'],
        'epoch_time_seconds': epoch_metrics['epoch_time_seconds'],
        'learning_rate': epoch_metrics['learning_rate'],
        'gpu_memory_mb': epoch_metrics['gpu_memory_mb'],
    },
    'final_metrics': {
        'train_accuracy': final_train_acc,
        'val_accuracy': final_val_acc,
        'test_accuracy': final_test_acc,
        'final_loss': epoch_metrics['train_loss'][-1],
        'total_training_time_seconds': epoch_metrics['total_training_time_seconds'],
    },
    'model_info': {
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
    },
    'system_info': {
        'device': str(device),
        'gpu_name': GPU_NAME,
        'cuda_version': torch.version.cuda if torch.cuda.is_available() else None,
        'pytorch_version': torch.__version__,
        'python_version': platform.python_version(),
        'platform': platform.platform(),
    },
    'timestamp': datetime.now().isoformat(),
}

with open(results_path, 'w') as f:
    json.dump(training_results, f, indent=2)

print(f"Training results saved to: {results_path}")

In [ ]:
# =============================================================================
# SAVE TRAINING CHART
# =============================================================================

plot_path = PLOTS_DIR / f"{EXPERIMENT_NAME}_training.png"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_list = epoch_metrics['epoch']

# Plot 1: Loss curve
ax1.plot(epochs_list, epoch_metrics['train_loss'], 'b-', linewidth=2, label='Train Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title(f'Training Loss - {EXPERIMENT_NAME}', fontsize=14)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, EPOCHS)

# Plot 2: Accuracy curves
ax2.plot(epochs_list, [acc*100 for acc in epoch_metrics['train_accuracy']], 
         'b-', linewidth=2, label='Train Accuracy')
ax2.plot(epochs_list, [acc*100 for acc in epoch_metrics['val_accuracy']], 
         'r-', linewidth=2, label='Val Accuracy')
ax2.axhline(y=final_test_acc*100, color='g', linestyle='--', linewidth=1.5, 
            label=f'Test Accuracy ({final_test_acc*100:.1f}%)')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title(f'Training/Validation Accuracy - {EXPERIMENT_NAME}', fontsize=14)
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, EPOCHS)
ax2.set_ylim(50, 100)

plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Training chart saved to: {plot_path}")
plt.show()

In [ ]:
# =============================================================================
# SAVE CSV SUMMARY
# =============================================================================

summary_path = RESULTS_DIR / f"{EXPERIMENT_NAME}_summary.csv"

summary_data = {
    'experiment_name': [EXPERIMENT_NAME],
    'distances': [str(DISTANCES)],
    'num_distances': [len(DISTANCES)],
    'total_samples': [TOTAL_SAMPLES],
    'epochs': [EPOCHS],
    'batch_size': [BATCH_SIZE],
    'learning_rate': [BEST_HYPERPARAMS['learning_rate']],
    'hidden_dim': [BEST_HYPERPARAMS['hidden_dim']],
    'num_layers': [BEST_HYPERPARAMS['num_layers']],
    'aggr': [BEST_HYPERPARAMS['aggr']],
    'dropout': [BEST_HYPERPARAMS['dropout']],
    'final_loss': [epoch_metrics['train_loss'][-1]],
    'train_accuracy': [final_train_acc],
    'val_accuracy': [final_val_acc],
    'test_accuracy': [final_test_acc],
    'training_time_minutes': [epoch_metrics['total_training_time_seconds'] / 60],
    'total_parameters': [total_params],
    'device': [str(device)],
    'gpu_name': [GPU_NAME],
    'timestamp': [datetime.now().isoformat()],
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_path, index=False)
print(f"Summary CSV saved to: {summary_path}")

# Display summary
print(f"\nSummary:")
print(summary_df.T.to_string(header=False))

## 8. Final Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE - {EXPERIMENT_NAME}")
print(f"{'='*70}")

print(f"\nConfiguration:")
print(f"  Distances: {DISTANCES}")
print(f"  Total samples: {TOTAL_SAMPLES:,}")
print(f"  Epochs: {EPOCHS}")

print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

print(f"\nFinal Results:")
print(f"  Train accuracy: {final_train_acc*100:.2f}%")
print(f"  Validation accuracy: {final_val_acc*100:.2f}%")
print(f"  Test accuracy: {final_test_acc*100:.2f}%")
print(f"  Final loss: {epoch_metrics['train_loss'][-1]:.4f}")
print(f"  Training time: {epoch_metrics['total_training_time_seconds']/60:.1f} minutes")

print(f"\nSaved Artifacts:")
print(f"  Model: {model_path}")
print(f"  Results JSON: {results_path}")
print(f"  Training chart: {plot_path}")
print(f"  Summary CSV: {summary_path}")

print(f"\n{'='*70}")

In [ ]:
# Display per-epoch metrics table
print(f"\nPer-Epoch Metrics:")
print(f"{'='*70}")

metrics_df = pd.DataFrame({
    'Epoch': epoch_metrics['epoch'],
    'Train Loss': epoch_metrics['train_loss'],
    'Train Acc': [f"{acc*100:.2f}%" for acc in epoch_metrics['train_accuracy']],
    'Val Acc': [f"{acc*100:.2f}%" for acc in epoch_metrics['val_accuracy']],
    'Time (s)': [f"{t:.1f}" for t in epoch_metrics['epoch_time_seconds']],
    'GPU (MB)': [f"{m:.0f}" for m in epoch_metrics['gpu_memory_mb']],
})

# Show first 5 and last 5 epochs
if len(metrics_df) > 10:
    print(metrics_df.head(5).to_string(index=False))
    print("...")
    print(metrics_df.tail(5).to_string(index=False))
else:
    print(metrics_df.to_string(index=False))